# Лабораторная работа № 6 · Разметка и подготовка данных

**CineMatch · ДГТУ · МИК21**

Задача: подготовить версионированный набор MovieLens, назначить метки релевантности 0/1/2, защитить временной split от утечек и проверить пилот.

**Инструкция:** [annotation_guideline.md](https://github.com/localzet/localzet/blob/main/education/dgtu/ml-creative-industries/lab06-labeling-cinematch/annotation_guideline.md).
**Паспорт:** [Data Card v1.2](https://github.com/localzet/localzet/blob/main/education/dgtu/ml-creative-industries/lab06-labeling-cinematch/data_card_update.md).
**Статистика пилота:** [annotation_results.md](https://github.com/localzet/localzet/blob/main/education/dgtu/ml-creative-industries/lab06-labeling-cinematch/annotation_results.md).

Доступ по публичной ссылке; данные скачиваются при выполнении ячеек (или укажите путь к локальному ZIP). Результаты вычисляются кодом. Второго независимого разметчика-человека нет: применяется альтернативная программная проверка.

In [ ]:
from urllib.request import urlretrieve
from pathlib import Path
import sys
BASE = "https://raw.githubusercontent.com/localzet/localzet/main/education/dgtu/ml-creative-industries/lab06-labeling-cinematch/"
for fname in ["prepare_dataset.py", "test_prepare.py", "data_manifest_v1.json", "pilot_labels_v1.csv"]:
    urlretrieve(BASE + fname, fname)
print("Код и зафиксированные артефакты загружены из публичного репозитория")

In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "unittest", "-v", "test_prepare"], check=True)

In [ ]:
# Скачивание официального архива или проверенного зеркала, контроль Git blob SHA.
subprocess.run([sys.executable, "prepare_dataset.py", "--output", "prepared"], check=True)

In [ ]:
import json
import pandas as pd
from IPython.display import display
manifest = json.loads(Path("prepared/manifest.json").read_text(encoding="utf-8"))
display(pd.DataFrame(manifest["class_counts_by_split"]).T)
display(pd.DataFrame.from_dict(manifest["split_counts"], orient="index", columns=["records"]))
display(pd.read_csv("prepared/pilot_labels.csv"))
print("IDF обучен только на", manifest["train_unique_movies"], "уникальных фильмах из train")

## Вывод

* Классы **0** (0.5–2.5), **1** (3.0–3.5), **2** (4.0–5.0) назначаются по неизменяемой инструкции. Ненаблюдаемый рейтинг — не 0.
* Связанные события одного пользователя с одинаковым timestamp сохраняются в одном split; предварительно разрешаются повторные `(userId,movieId)`.
* IDF обучается на **train**, изменения test не должны менять веса.
* Программное совпадение двух реализаций не эквивалентно коэффициенту согласия двух людей.
* Файлы prepared содержат исходные псевдонимные ID; **не публикуйте их в открытом репозитории**.

Дальнейшее использование: baseline и сравнение моделей в ЛР 7. Источник: Harper & Konstan (2015), MovieLens Datasets; [GroupLens terms](https://files.grouplens.org/datasets/movielens/ml-latest-small-README.html).